# 2.1 LSTM with target-aware attention — initial relevance scoring

**Experiment ID:** `2.1_lstm` · **Track:** neural networks · **Stage:** base click
scoring of the recorded impression slate (no retrieval, no reranking).

**Quick start (Colab, GPU runtime):** run all cells top to bottom. Defaults train the
main model on the full shared splits, train two control variants, export all six
CSVs, persist the run to Drive, and reload it from the Drive copy. For a fast
pipeline check, set `SMOKE_TEST = True` in Section 2 first. To export from a saved
model without training, set `TRAIN_MODEL = False` and `LOAD_RUN_ID`.

Code lives next to this notebook in `lstm_data.py` (data, metrics, exports; no
PyTorch) and `lstm_model.py` (network, batching, training). Both are copied into
every saved model bundle, so the bundle can be reloaded without this repository.

## 1. Business hypothesis and experiment overview

| Item | Detail |
| --- | --- |
| ID / owner | `2.1_lstm` / see `OWNER` in Section 2 |
| Model | Chronological history → LSTM → candidate-as-query masked attention → MLP head (PyTorch) |
| Business objective | Interim report §1.1 and §1.2.3: relevant recommendations that support continued reading and exploration; this model supplies the **initial relevance score** for every candidate, and the shared stage reranks for continuity, freshness and diversity |
| Output | One finite `score` (BCE logit) per recorded candidate; `click_probability = sigmoid(score)`, uncalibrated |
| Primary / secondary metric | Mean impression AUC (selection) / mean impression nDCG@5 |

**Hypothesis.** A reader's interests are not one static profile. Within a news
history, recent reads, repeat visits and reading depth say which interest is
active right now, and different candidates relate to different parts of that
history. So:

1. **Target-aware attention.** Weighting history by relevance *to each candidate*
   (a football article attends to past sport reads, a crime story to past crime
   reads) should rank candidates better than one fixed summary of the user.
2. **Order.** Reading the history in chronological order with an LSTM should beat
   the same events in a random order, if the sequence carries drift and session
   information beyond each event's own timestamp.
3. **Engagement and content.** Read time and scroll depth from past reads, and
   frozen text and image vectors, should let the model score articles it has
   never seen, which a news catalogue produces constantly.

**Focused validation checks** (development data, same candidate rows):

| Comparison | Variant | Evidence that supports keeping the more complex choice |
| --- | --- | --- |
| Target-aware attention vs simpler summary | `attention` vs `last_state` (final LSTM state, same for all candidates) | Paired mean-impression-AUC gain whose user-cluster bootstrap 95% CI excludes 0, plus attention concentrated on related history (category-match lift > 1) |
| Ordered vs shuffled history | `attention` vs `attention_shuffled` (same events and recency, fixed random order, no order-derived gaps) | Same rule. If the CI includes 0, ordering adds nothing measurable beyond per-event recency, and a cheaper pooled model may be enough |

A third control, `attention_no_lstm` (attention over per-event inputs with no
recurrence), is available but off by default.

**What this evidence cannot show.** These are offline click-ranking results on
logged slates. A gain supports a *relevance* claim only. It does not show more
continued reading or retention, which need session-continuation outcomes or
online tests (shared evaluation stage). Attention weights are diagnostics of what
the model uses, not causal explanations of reader behaviour.

**Leakage controls.** Inputs are limited to information available at impression
time:
- Histories come from each split's fixed pre-period history file. Every event is
  asserted to precede the split's first impression. Later in-period clicks are not
  appended.
- Current or next read time and scroll, `total_inviews`, `total_pageviews` and
  `total_read_time` are never read.
- The official test split has no labels and is never used for any decision.

### Design decisions and what would support them

| Decision | Starting choice | Rationale / evidence to check after the run |
| --- | --- | --- |
| History source | Raw `history.parquet` lists (article, time, read time, scroll) per split; the engineered table has only aggregates | Preflight shows aligned lists, chronological sort, cutoff respected |
| History length | Most recent **50** events (configurable) | Covers the median EB-NeRD history (report §2.1.2: median 80–92 on LARGE) while bounding cost. Not tuned; Section 6 reports AUC by history length |
| Empty history | Learned "empty history" context vector + `history_empty` flag | Rare in this data. Counts reported in Section 3 |
| Unknown / missing articles | Kept; category/type "missing"/"unknown" indices; zero vector **plus** missing flags for absent text/image vectors | Coverage table in Section 4 |
| Article representation | Category, type, topics (shared train vocabulary), premium, body length, sentiment, frozen Gemini text (768-d) and EB-NeRD image (1024-d) vectors, projected **separately** | Scores unseen articles. Learned article-ID embedding is optional (off), so IDs are never the only route |
| Recency | Log hours from each history event to the impression as an attention bias; log age of each article at read time and inter-event gap as LSTM inputs | DeNA/Tom3TK found time features strongest among leakage-free signals |
| Candidate freshness | Shared `article_age_h` (null if unknown or published after the impression) + within-slate age rank | Same |
| Objective | BCE on logits over every candidate (multiple clicks allowed); not the course's SmoothL1 rating loss | Click target is binary |
| Optimiser | AdamW (lr 1e-3, wd 1e-5), grad-clip 1.0, ReduceLROnPlateau on val AUC, early stopping (patience 2), AMP on GPU | Standard, course-connected. AMP only speeds up training |
| Selection | Best validation mean impression AUC; ties keep the earlier epoch | Common protocol. Validation is **development data** because it is used for early stopping and any tuning |

## 2. Setup and run configuration

In [ ]:
# Environment: mount Drive (Colab), locate the experiment code, record versions.
import os, sys, subprocess, platform, json, shutil, time, math, importlib, importlib.util
from pathlib import Path
try:
    from IPython.display import display
except ImportError:  # plain Python
    display = print
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
if IN_COLAB and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

# Where to find lstm_data.py / lstm_model.py. The repository is cloned if needed.
REPO_URL = "https://github.com/KoalaLyfestyle/BT4222-Final-Project.git"
REPO_BRANCH = "claude/serene-hawking-gkuqc7"   # switch to "main" after merge
CODE_DIR_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "Experiments" / "2.1_lstm",
    Path("/content/BT4222-Final-Project/Experiments/2.1_lstm"),
]
CODE_DIR = next((p for p in CODE_DIR_CANDIDATES
                 if (p / "lstm_data.py").is_file() and (p / "lstm_model.py").is_file()), None)
if CODE_DIR is None and IN_COLAB:
    target = Path("/content/BT4222-Final-Project")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
                        REPO_URL, str(target)], check=True)
    CODE_DIR = target / "Experiments" / "2.1_lstm"
if CODE_DIR is None or not (CODE_DIR / "lstm_data.py").is_file():
    raise FileNotFoundError(
        "Cannot find lstm_data.py/lstm_model.py. Clone the repository or copy the two "
        "files next to this notebook and add that folder to CODE_DIR_CANDIDATES.")
sys.path.insert(0, str(CODE_DIR))

def _git(*args):
    try:
        return subprocess.run(["git", "-C", str(CODE_DIR), *args], capture_output=True,
                              text=True, check=True).stdout.strip()
    except Exception:
        return None
GIT_INFO = {"commit": _git("rev-parse", "HEAD"),
            "branch": _git("rev-parse", "--abbrev-ref", "HEAD"),
            "uncommitted_changes": bool(_git("status", "--porcelain")) if _git("rev-parse", "HEAD") else None,
            "code_dir": str(CODE_DIR)}
print("Code:", GIT_INFO)

In [ ]:
# ---------------------------------------------------------------- run configuration
EXPERIMENT_ID = "2.1_lstm"
OWNER = "Winston Ho Rui Wei"          # edit if someone else runs this
SEED = 42
FEATURE_CASE = "raw"   # only keys, labels and a few context columns are read; see Section 3

# Data root. The shared folder is owned by another account: add a Drive shortcut
# ("Add shortcut to Drive") so it appears under MyDrive, then point to it here.
DRIVE_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder"),
    Path("/content/drive/MyDrive/BT4222 Shared Final Folder"),
]
DATA_ROOT = None        # set explicitly to override, e.g. a mini sample folder
PATH_OVERRIDES = {}     # {key: path} for any input in lstm_data.PATH_CANDIDATES

# Train / load switches
TRAIN_MODEL = True       # False: load LOAD_RUN_ID from PERSIST_ROOT and export
LOAD_RUN_ID = None       # e.g. "20261104_0930_seed42"
RESUME_RUN_ID = None     # resume an interrupted training run from its Drive copy

# Development options
SMOKE_TEST = False       # True: small whole-user sample, 1 epoch, labelled as smoke
SMOKE_USER_FRACTION = 0.05
DEV_USER_FRACTION = None # e.g. 0.25 for a quicker non-smoke development run
CHECK_AGAINST_BEHAVIOURS = True   # confirm feature rows == exploded in-view slates
HASH_INPUTS = True       # sha256 of every input (a few minutes on Drive)

# History
HISTORY_MAX_LEN = 50

# Model (starting values; defensible, not claimed optimal)
MODEL_VARIANT = "attention"
MODEL_HPARAMS = dict(
    article_dim=128, category_dim=16, type_dim=8, topic_dim=16, text_dim=64, image_dim=32,
    lstm_hidden=128, lstm_layers=1, attention_dim=64, interaction_dim=64,
    head_hidden=(256, 128), dropout=0.2, embedding_dropout=0.1,
    use_id_embedding=False, id_dim=16,
)
ID_EMBEDDING_MIN_COUNT = 5   # only used when use_id_embedding=True
TRAIN_HPARAMS = dict(
    max_epochs=8, batch_impressions=256, learning_rate=1e-3, weight_decay=1e-5,
    grad_clip=1.0, early_stopping_patience=2, lr_factor=0.5, lr_patience=0,
    min_lr=1e-5, history_dropout=0.0, use_amp=True, eval_batch_impressions=256,
    max_train_impressions_per_epoch=None,
)

# Comparisons (Section 6): same data, hyperparameters and budget as the main model
RUN_ABLATIONS = True
ABLATION_VARIANTS = ["last_state", "attention_shuffled"]   # + "attention_no_lstm"
SHUFFLE_SEED = 2024
BOOTSTRAP_SAMPLES = 1000

# Optional Optuna search (development validation; never test)
RUN_OPTUNA = False
APPLY_BEST_TRIAL = False      # copy the best trial's values into the configs above
OPTUNA_STUDY_NAME = "lstm_attention_v1"
OPTUNA_N_TRIALS = 12
OPTUNA_TIMEOUT_MIN = 150
OPTUNA_EPOCHS = 3
OPTUNA_TRAIN_IMPRESSIONS_PER_EPOCH = 60_000

# Exports and persistence
EXPORT_CLICK_PROBABILITY = True
PERSIST_TO_DRIVE = True
PERSIST_DURING_TRAINING = True
RELOAD_CHECK_IMPRESSIONS = 300
RELOAD_TOLERANCE = 1e-4       # max |logit difference| (fp32 GPU/CPU nondeterminism)

if SMOKE_TEST:
    TRAIN_HPARAMS.update(max_epochs=1, batch_impressions=64)
    OPTUNA_N_TRIALS, OPTUNA_EPOCHS = 2, 1

In [ ]:
# Resolve paths, run folders, seed and device.
import numpy as np
import pandas as pd
import pyarrow as pa
import torch
import lstm_data as D
import lstm_model as M
importlib.reload(D); importlib.reload(M)

if DATA_ROOT is None:
    DATA_ROOT = next((p for p in DRIVE_ROOT_CANDIDATES if p.is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(f"No shared data folder found in {DRIVE_ROOT_CANDIDATES}. "
                            "Add a Drive shortcut or set DATA_ROOT.")
DATA_ROOT = Path(DATA_ROOT)
PATHS, PATH_NOTES = D.resolve_paths(DATA_ROOT, PATH_OVERRIDES)
for key, path in PATHS.items():
    print(f"{key:30s} {path}")
for note in PATH_NOTES:
    print("NOTE:", note)
IS_MINI_SAMPLE = (DATA_ROOT / "sample_manifest.json").is_file()
if IS_MINI_SAMPLE:
    print("WARNING: DATA_ROOT is the mini implementation sample; results are not benchmarks.")

PERSIST_ROOT = DATA_ROOT / "Experiments" / EXPERIMENT_ID
LOCAL_ROOT = (Path("/content/bt4222_outputs") / EXPERIMENT_ID if IN_COLAB
              else CODE_DIR / "outputs")
stamp = time.strftime("%Y%m%d_%H%M")
if RESUME_RUN_ID and not TRAIN_MODEL:
    raise ValueError("RESUME_RUN_ID is for training; use LOAD_RUN_ID with TRAIN_MODEL=False")
if not TRAIN_MODEL:
    # Saved settings that shape the inputs must be known before Section 3.
    LOAD_DIR = PERSIST_ROOT / str(LOAD_RUN_ID) / "model"
    if not LOAD_DIR.is_dir():
        raise FileNotFoundError(f"Saved model bundle not found: {LOAD_DIR}")
    HISTORY_MAX_LEN = json.loads((LOAD_DIR / "preprocessing.json").read_text())["history_max_len"]
    MODEL_VARIANT = json.loads((LOAD_DIR / "model_config.json").read_text())["variant"]
    print(f"Load-only: using saved HISTORY_MAX_LEN={HISTORY_MAX_LEN}, variant={MODEL_VARIANT}")
if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
elif not TRAIN_MODEL:
    if not LOAD_RUN_ID:
        raise ValueError("TRAIN_MODEL=False needs LOAD_RUN_ID")
    RUN_ID = f"{stamp}_seed{SEED}_load"
else:
    RUN_ID = f"{stamp}_seed{SEED}" + ("_smoke" if SMOKE_TEST else "") + \
             ("_mini" if IS_MINI_SAMPLE else "") + ("_dev" if DEV_USER_FRACTION else "")
RUN_DIR = LOCAL_ROOT / RUN_ID
PERSIST_RUN_DIR = PERSIST_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_ID and PERSIST_RUN_DIR.is_dir() and not (RUN_DIR / "model_training").exists():
    shutil.copytree(PERSIST_RUN_DIR, RUN_DIR, dirs_exist_ok=True)
    print("Copied the Drive run folder locally for resumption.")

LOG_PATH = RUN_DIR / "logs" / "notebook_log.txt"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def log(*parts):
    message = " ".join(str(p) for p in parts)
    print(message)
    with open(LOG_PATH, "a") as handle:
        handle.write(time.strftime("%H:%M:%S ") + message + "\n")

M.set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VERSIONS = {"python": platform.python_version(), "torch": torch.__version__,
            "numpy": np.__version__, "pandas": pd.__version__, "pyarrow": pa.__version__,
            "cuda": torch.version.cuda, "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
log("Run:", RUN_ID, "| local:", RUN_DIR, "| persistent:", PERSIST_RUN_DIR)
log("Versions:", VERSIONS)
if DEVICE.type != "cuda" and not SMOKE_TEST:
    log("WARNING: no GPU; full training will be slow. Runtime > Change runtime type > GPU.")

## 3. Load and check shared data

**Candidate rows** come from the shared engineered table
`feature_engineering/{split}_features.parquet`. Only these columns are read:
- the keys and the label;
- `impression_time`;
- the impression-context columns `hour`, `day_of_week`, `slate_size`,
  `is_sso_user`, `is_subscriber`, `is_front_page` and the train-vocabulary device
  one-hots;
- the shared candidate freshness fields `article_age_h` and `article_age_missing`.

Each impression is made contiguous; its in-view order is kept. No candidate is
added or removed. The 200 engineered history aggregates are deliberately **not**
inputs here. The model must learn from the ordered history itself, which also
keeps the attention-versus-summary comparison clean. `FEATURE_CASE` is recorded,
and the raw file is the only one read.

In [ ]:
FRAMES, OFFSETS, TABLE_INFO, SAMPLING = {}, {}, {}, {}
for split in D.SPLITS:
    frame, info = D.load_candidate_table(PATHS[f"features_{split}"], split)
    offsets = info.pop("offsets")
    fraction = SMOKE_USER_FRACTION if SMOKE_TEST else DEV_USER_FRACTION
    if CHECK_AGAINST_BEHAVIOURS and PATHS.get(f"behaviors_{split}"):
        info["behaviour_cross_check"] = D.cross_check_behaviours(
            frame, PATHS[f"behaviors_{split}"], split)
        check = info["behaviour_cross_check"]
        assert check["feature_rows_not_in_behaviours"] == 0, check
        assert check["behaviour_candidates_not_in_features"] == 0, check
        assert check.get("label_mismatches", 0) == 0, check
    if fraction:
        frame, offsets, rule = D.select_development_sample(frame, offsets, fraction, SEED)
        SAMPLING[split] = rule
    FRAMES[split], OFFSETS[split], TABLE_INFO[split] = frame, offsets, info

DEVICE_COLUMNS = TABLE_INFO["train"]["device_columns"]
for split in D.SPLITS:
    assert TABLE_INFO[split]["device_columns"] == DEVICE_COLUMNS, f"{split} device columns differ"
summary = pd.DataFrame({s: {k: v for k, v in TABLE_INFO[s].items()
                            if k not in ("device_columns", "behaviour_cross_check")}
                        for s in D.SPLITS}).T
display(summary)
for split in D.SPLITS:
    if "behaviour_cross_check" in TABLE_INFO[split]:
        log(split, "behaviour cross-check:", TABLE_INFO[split]["behaviour_cross_check"])
if SAMPLING:
    log("DEVELOPMENT SAMPLE (whole users):", SAMPLING)
assert TABLE_INFO["test"]["labelled"] is False

**Article catalogue and frozen vectors (preflight).**
- **Catalogue.** The train/validation catalogue takes precedence; the test
  catalogue adds test-only articles (first-wins, as in Step 3).
- **Vectors.** The vector columns are actually decoded. The checks:
  - dimensions (768 text / 1024 image);
  - finite values;
  - flag ↔ null agreement;
  - duplicate IDs;
  - identical vectors for IDs present in both lookups.
- **Missing vs unreadable.** A legitimately missing vector stays missing (with a
  flag). An unreadable file stops the notebook with a diagnosis; it is not
  replaced with zeros.

In [ ]:
VOCAB = json.loads(PATHS["feature_vocabulary"].read_text())
CATALOGUE, CATALOGUE_INFO = D.load_catalogue([PATHS["articles_train_val"], PATHS["articles_test"]])
log("Catalogue:", CATALOGUE_INFO)

def expected_dim(kind, default):
    for key in ("embedding_metadata_train_val", "embedding_metadata_test"):
        if PATHS.get(key):
            meta = json.loads(PATHS[key].read_text())
            value = meta.get(f"{kind}_dimension")
            if value and value != default:
                raise ValueError(f"{key}: {kind} dimension {value} != {default}")
    return default
TEXT_DIM, IMAGE_DIM = expected_dim("text", 768), expected_dim("image", 1024)

VECTOR_INFO, VECTOR_SOURCES = {}, {"text": [], "image": []}
for kind, column, flag, dim in (("text", "text_embedding", "has_text_embedding", TEXT_DIM),
                                ("image", "image_embedding", "has_image_embedding", IMAGE_DIM)):
    for scope in ("train_val", "test"):
        ids, matrix, available, info = D.decode_vectors(PATHS[f"{kind}_{scope}"], column, flag, dim)
        VECTOR_SOURCES[kind].append((ids, matrix, available))
        VECTOR_INFO[f"{kind}_{scope}"] = info
        log(f"{kind:5s} {scope:9s}", {k: v for k, v in info.items() if k != "path"})
TEXT_LOOKUP, text_union = D.union_vectors(VECTOR_SOURCES["text"])
IMAGE_LOOKUP, image_union = D.union_vectors(VECTOR_SOURCES["image"])
for name, union in (("text", text_union), ("image", image_union)):
    assert union["overlap_vector_mismatches"] == 0, f"{name}: overlapping IDs disagree"
    VECTOR_INFO[f"{name}_union"] = {k: v for k, v in union.items() if k != "known_ids"}
    log(name, "union:", VECTOR_INFO[f"{name}_union"])
del VECTOR_SOURCES

**Chronological histories.** For each split, its own fixed `history.parquet`:
- **Alignment.** The four lists (article ID, timestamp, read time, scroll %) must
  have equal lengths.
- **Invalid events.** Events with a null ID or timestamp are dropped and counted.
  Invalid read/scroll values become missing, with flags.
- **Cutoff.** Every event must be strictly before the split's earliest impression;
  otherwise the notebook stops.
- **Ordering.** Events are sorted chronologically; the original list position
  breaks ties. Repeat reads of the same article are kept as separate events.
- **Truncation.** The most recent `HISTORY_MAX_LEN` events are kept, oldest
  first, then right-padded with masks.
- **Users without a history row.** These map to an empty row and the model's
  empty-history fallback.

In [ ]:
HISTORIES = {}
for split in D.SPLITS:
    split_start = pd.Timestamp(FRAMES[split]["impression_time"].min())
    # Cutoff uses the split's earliest impression in the *full* table when sampling.
    if SAMPLING:
        import pyarrow.parquet as pq
        times = pq.read_table(PATHS[f"features_{split}"], columns=["impression_time"])
        split_start = pd.Timestamp(times["impression_time"].to_numpy().min())
    HISTORIES[split] = D.build_history(PATHS[f"history_{split}"], split_start,
                                       HISTORY_MAX_LEN, FRAMES[split]["user_id"].unique())
    log(split, "history:", HISTORIES[split].stats)

## 4. Prepare model inputs

**Fitted on train only** (after any development sampling):
- standardisation constants for log candidate age, log slate size, log history
  length, log gap to impression, log read time, scroll fraction, inter-event gap,
  body length and sentiment;
- the optional article-ID vocabulary.

**Reused unchanged:** the shared Step 3 train vocabulary (category, article type,
topics).

In load-only mode every fitted value is read from the saved bundle; nothing is
refitted.

In [ ]:
def referenced_ids(split):
    hist = HISTORIES[split]
    mask = np.arange(hist.article_ids.shape[1])[None, :] < hist.lengths[:, None]
    return np.concatenate([FRAMES[split]["candidate_article_id"].to_numpy(),
                           hist.article_ids[mask]])

if TRAIN_MODEL:
    train_ids = referenced_ids("train")
    PREPROCESSING = {
        "numeric_stats": D.fit_numeric_stats(FRAMES["train"], HISTORIES["train"], OFFSETS["train"]),
        "article_stats": D.fit_article_stats(CATALOGUE, train_ids),
        "history_max_len": HISTORY_MAX_LEN, "device_columns": DEVICE_COLUMNS,
        "feature_vocabulary": VOCAB, "text_dim": TEXT_DIM, "image_dim": IMAGE_DIM,
        "max_topics": 8, "shuffle_seed": SHUFFLE_SEED,
    }
    ID_VOCAB = (D.fit_id_vocab(train_ids, ID_EMBEDDING_MIN_COUNT)
                if MODEL_HPARAMS["use_id_embedding"] else {})
    seen_train_ids = set(np.unique(train_ids).tolist())
else:
    PREPROCESSING = json.loads((LOAD_DIR / "preprocessing.json").read_text())
    mappings = np.load(LOAD_DIR / "mappings.npz")
    ID_VOCAB = dict(zip(mappings["id_vocab_ids"].tolist(), mappings["id_vocab_rows"].tolist()))
    seen_train_ids = set(mappings["train_referenced_ids"].tolist())
    assert HISTORIES["train"].article_ids.shape[1] == PREPROCESSING["history_max_len"]
    assert PREPROCESSING["feature_vocabulary"] == VOCAB, "shared vocabulary changed since training"
    assert PREPROCESSING["device_columns"] == DEVICE_COLUMNS, "device columns changed"
    log("Loaded preprocessing from", LOAD_DIR)

all_ids = np.concatenate([referenced_ids(s) for s in D.SPLITS])
ARTICLES, ARTICLE_INFO = D.build_article_table(
    CATALOGUE, all_ids, VOCAB, PREPROCESSING["article_stats"], TEXT_LOOKUP, IMAGE_LOOKUP,
    TEXT_DIM, IMAGE_DIM, PREPROCESSING["max_topics"], ID_VOCAB)
log("Article table:", ARTICLE_INFO)

def make_arrays(variant):
    shuffled = variant == "attention_shuffled"
    out = {}
    for split in D.SPLITS:
        hist = HISTORIES[split]
        if shuffled:
            hist = D.shuffled_history(hist, PREPROCESSING["shuffle_seed"] + D.SPLITS.index(split))
        out[split] = D.build_split_arrays(split, FRAMES[split], OFFSETS[split], hist, ARTICLES,
                                          PREPROCESSING["numeric_stats"], DEVICE_COLUMNS,
                                          disable_order_features=shuffled)
    return out

ARRAYS = {"ordered": make_arrays("attention")}
MAIN_KEY = "shuffled" if MODEL_VARIANT == "attention_shuffled" else "ordered"
if MAIN_KEY not in ARRAYS:
    ARRAYS[MAIN_KEY] = make_arrays(MODEL_VARIANT)
coverage = {s: D.coverage_report(ARRAYS["ordered"][s], ARTICLES) for s in D.SPLITS}
display(pd.DataFrame({(s, part): v for s in coverage for part, v in coverage[s].items()}).T)
for split in D.SPLITS:
    a = ARRAYS["ordered"][split]
    assert np.isfinite(a.imp_context).all() and np.isfinite(a.cand_features).all()
    assert np.isfinite(a.hist_features).all()
    assert len(a.cand_article_id) == len(FRAMES[split])
log("Context features:", ARRAYS["ordered"]["train"].context_names)
log("Candidate features:", ARRAYS["ordered"]["train"].candidate_names)
log("Event features:", ARRAYS["ordered"]["train"].event_names)
assert ARRAYS["ordered"]["train"].event_names[M.GAP_PREV_FEATURE] == "log_gap_prev_h_z"
assert ARRAYS["ordered"]["train"].event_names[M.GAP_PREV_MISSING] == "gap_prev_missing"

In [ ]:
# Device tensors (article vectors stay in one lookup table; candidates hold indices).
ARTICLE_TENSORS = M.article_table_tensors(ARTICLES, DEVICE)
TENSORS = {key: {s: M.SplitTensors(ARRAYS[key][s], DEVICE) for s in D.SPLITS}
           for key in ARRAYS}
VAL = ARRAYS["ordered"]["validation"]

def evaluate_validation(scores, arrays=VAL, per_impression=False):
    return D.impression_metrics(arrays.offsets, arrays.labels, scores, k=5,
                                return_per_impression=per_impression)

def model_config(variant, overrides=None):
    sample = ARRAYS["ordered"]["train"]
    stats = PREPROCESSING["numeric_stats"]["log_gap_to_impression_h"]
    params = dict(MODEL_HPARAMS, **(overrides or {}))
    params["head_hidden"] = tuple(params["head_hidden"])
    return M.ModelConfig(
        variant=variant, n_categories=len(VOCAB["category"]) + 3,
        n_types=len(VOCAB["article_type"]) + 3, n_topics=len(VOCAB["topics"]) + 2,
        n_article_numeric=ARTICLES.numeric.shape[1], text_dim_in=TEXT_DIM, image_dim_in=IMAGE_DIM,
        n_id_buckets=max(len(ID_VOCAB) + 1, 1), n_event_features=sample.hist_features.shape[-1],
        n_candidate_features=sample.cand_features.shape[1],
        n_context_features=sample.imp_context.shape[1],
        gap_mean=stats["mean"], gap_std=stats["std"], **params)

if DEVICE.type == "cuda":
    log(f"GPU memory allocated: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

## 5. Define and train / load model

**Architecture** (see `lstm_model.py`):
1. **Article encoder**, shared by history events and candidates. It concatenates:
   - category, type and topic embeddings (shared train vocabulary);
   - numeric metadata and missing flags;
   - a projection of the frozen text vector;
   - a separate projection of the frozen image vector.

   An MLP and LayerNorm then give a 128-d article vector. The supplied vectors are
   frozen lookups; only the projections train.
2. **History encoder.** Each event is the article vector plus log read time,
   scroll and missing flags, log gap since the previous event, and log article age
   when read. Events pass through a linear+ReLU layer, then a 1-layer LSTM (packed
   sequences, oldest → newest). Padding is masked.
3. **Target-aware attention.** The candidate vector becomes the query and the LSTM
   states the keys. The score is the scaled dot product plus a learned
   recency bias × standardised log(hours from event to impression). A masked
   softmax gives weights, and the candidate-specific context is the weighted sum of
   states. With an empty history the model uses a learned fallback vector instead.
4. **Head.** An MLP (256 → 128 → 1, dropout) over:
   - the candidate vector and the context;
   - the elementwise product of their projections and its scaled dot product;
   - candidate freshness (log age and within-slate age rank);
   - impression context (time, device, subscriber, slate size, history length).

   Output: a click logit. Training loss is BCE over every candidate.

The course recipe regressed movie ratings with SmoothL1; here the target is a
binary click, so the loss is `BCEWithLogitsLoss`, and `sigmoid(logit)` is exported
as an uncalibrated click probability.

In [ ]:
MAIN_CONFIG = model_config(MODEL_VARIANT)
_preview = M.build_model(MAIN_CONFIG, ARTICLE_TENSORS, DEVICE)
log("Main model:", MAIN_CONFIG.variant, M.count_parameters(_preview))
print(_preview)
del _preview

### Optional: bounded Optuna search (off by default)

- **Objective:** maximise validation mean impression AUC on development data.
  Test is never touched.
- **Sampler and pruner:** TPE (seeded) with a MedianPruner over epochs.
- **Budget:** `OPTUNA_N_TRIALS` trials, each at most `OPTUNA_EPOCHS` epochs on
  `OPTUNA_TRAIN_IMPRESSIONS_PER_EPOCH` sampled training impressions.
- **Storage:** a local SQLite file. A consistent snapshot (SQLite backup API) and
  `trials.csv` are copied to Drive after every trial.
- **Resuming:** rerunning copies the Drive snapshot back and continues the study.
  Trials left `RUNNING` by a crash stay marked as such.
- **Not searched:** history length, because it would require rebuilding the
  sequences.

In [ ]:
TUNING_SUMMARY = None
if RUN_OPTUNA and TRAIN_MODEL:
    try:
        import optuna
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)
        import optuna
    import sqlite3
    tuning_local = Path("/content/optuna" if IN_COLAB else RUN_DIR / "optuna_local")
    tuning_local.mkdir(parents=True, exist_ok=True)
    tuning_drive = PERSIST_ROOT / "tuning" / OPTUNA_STUDY_NAME
    db_local = tuning_local / f"{OPTUNA_STUDY_NAME}.db"
    db_drive = tuning_drive / f"{OPTUNA_STUDY_NAME}.db"
    if db_drive.is_file() and not db_local.is_file():
        shutil.copy2(db_drive, db_local)
        log("Resuming study from Drive snapshot", db_drive)

    def snapshot(study, trial=None):
        tuning_drive.mkdir(parents=True, exist_ok=True)
        tmp = tuning_local / "snapshot.db"
        with sqlite3.connect(db_local) as src, sqlite3.connect(tmp) as dst:
            src.backup(dst)
        shutil.copy2(tmp, db_drive)
        study.trials_dataframe().to_csv(tuning_drive / "trials.csv", index=False)

    def objective(trial):
        overrides = dict(
            lstm_hidden=trial.suggest_categorical("lstm_hidden", [64, 128, 256]),
            attention_dim=trial.suggest_categorical("attention_dim", [32, 64, 128]),
            article_dim=trial.suggest_categorical("article_dim", [64, 128]),
            dropout=trial.suggest_float("dropout", 0.1, 0.4),
        )
        train_over = dict(
            learning_rate=trial.suggest_float("learning_rate", 3e-4, 3e-3, log=True),
            weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
            history_dropout=trial.suggest_float("history_dropout", 0.0, 0.3),
        )
        M.set_seed(SEED + trial.number)
        model = M.build_model(model_config(MODEL_VARIANT, overrides), ARTICLE_TENSORS, DEVICE)
        cfg = M.TrainConfig(**dict(TRAIN_HPARAMS, **train_over, max_epochs=OPTUNA_EPOCHS,
                                   max_train_impressions_per_epoch=OPTUNA_TRAIN_IMPRESSIONS_PER_EPOCH,
                                   seed=SEED + trial.number))
        _, best = M.train_model(model, TENSORS[MAIN_KEY]["train"], TENSORS[MAIN_KEY]["validation"],
                                evaluate_validation, cfg, tuning_local / f"trial_{trial.number}",
                                resume=False, trial=trial, log=log)
        shutil.rmtree(tuning_local / f"trial_{trial.number}", ignore_errors=True)
        trial.set_user_attr("best_epoch", best["epoch"])
        return best["auc"]

    study = optuna.create_study(
        study_name=OPTUNA_STUDY_NAME, storage=f"sqlite:///{db_local}", direction="maximize",
        load_if_exists=True, sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1))
    remaining = OPTUNA_N_TRIALS - len([t for t in study.trials if t.state.is_finished()])
    if remaining > 0:
        study.optimize(objective, n_trials=remaining, timeout=OPTUNA_TIMEOUT_MIN * 60,
                       callbacks=[snapshot], gc_after_trial=True)
    snapshot(study)
    TUNING_SUMMARY = {
        "study_name": OPTUNA_STUDY_NAME, "objective": "max validation mean impression AUC "
        "(development data)", "sampler": "TPESampler(seed=SEED)",
        "pruner": "MedianPruner(n_startup_trials=3, n_warmup_steps=1)",
        "budget": {"n_trials": OPTUNA_N_TRIALS, "timeout_min": OPTUNA_TIMEOUT_MIN,
                   "epochs": OPTUNA_EPOCHS,
                   "train_impressions_per_epoch": OPTUNA_TRAIN_IMPRESSIONS_PER_EPOCH},
        "best_value": study.best_value, "best_params": study.best_params,
        "best_trial": study.best_trial.number, "snapshot": str(db_drive),
        "trials_csv": str(tuning_drive / "trials.csv"), "optuna_version": optuna.__version__,
    }
    log("Optuna:", TUNING_SUMMARY)
    display(study.trials_dataframe())
    if APPLY_BEST_TRIAL:
        p = study.best_params
        for key in ("lstm_hidden", "attention_dim", "article_dim", "dropout"):
            MODEL_HPARAMS[key] = p[key]
        for key in ("learning_rate", "weight_decay", "history_dropout"):
            TRAIN_HPARAMS[key] = p[key]
        MAIN_CONFIG = model_config(MODEL_VARIANT)
        log("Applied best trial to MODEL_HPARAMS/TRAIN_HPARAMS")
else:
    print("Optuna search skipped (RUN_OPTUNA=False or load-only mode).")

In [ ]:
# Train the main model (or load it). Recovery checkpoints are copied to Drive each epoch.
def persist_training(local_dir, subdir):
    if not (PERSIST_TO_DRIVE and PERSIST_DURING_TRAINING):
        return None
    def callback(epoch, row):
        shutil.copytree(local_dir, PERSIST_RUN_DIR / subdir, dirs_exist_ok=True)
    return callback

TRAIN_CONFIG = M.TrainConfig(**dict(TRAIN_HPARAMS, seed=SEED))
if TRAIN_MODEL:
    M.set_seed(SEED)
    model = M.build_model(MAIN_CONFIG, ARTICLE_TENSORS, DEVICE)
    main_dir = RUN_DIR / "model_training"
    t0 = time.time()
    HISTORY, BEST = M.train_model(model, TENSORS[MAIN_KEY]["train"],
                                  TENSORS[MAIN_KEY]["validation"], evaluate_validation,
                                  TRAIN_CONFIG, main_dir, resume=bool(RESUME_RUN_ID),
                                  on_epoch_end=persist_training(main_dir, "model_training"), log=log)
    TRAIN_SECONDS = time.time() - t0
    log(f"Selected epoch {BEST['epoch']} (validation AUC {BEST['auc']:.4f}); "
        f"training {TRAIN_SECONDS / 60:.1f} min")
else:
    MAIN_CONFIG = M.ModelConfig.from_json(json.loads((LOAD_DIR / "model_config.json").read_text()))
    model = M.build_model(MAIN_CONFIG, ARTICLE_TENSORS, DEVICE)
    model.load_state_dict(torch.load(LOAD_DIR / "model_state.pt", map_location=DEVICE,
                                     weights_only=True))
    source_metrics = json.loads((PERSIST_ROOT / LOAD_RUN_ID / "metrics.json").read_text())
    HISTORY = json.loads((PERSIST_ROOT / LOAD_RUN_ID / "training_history.json").read_text())
    BEST = {"epoch": source_metrics["selection"]["selected_epoch"], "auc": None}
    TRAIN_SECONDS = None
    log("Loaded model from", LOAD_DIR)
model.eval()
pd.DataFrame(HISTORY)

In [ ]:
# Comparison variants: identical data, hyperparameters, budget and seed.
ABLATIONS = {}
if RUN_ABLATIONS and TRAIN_MODEL:
    for variant in ABLATION_VARIANTS:
        if variant == MODEL_VARIANT:
            continue
        ab_dir = RUN_DIR / "ablations" / variant
        if (ab_dir / "summary.json").is_file():   # finished before an interruption
            saved = json.loads((ab_dir / "summary.json").read_text())
            per = np.load(ab_dir / "validation_per_impression.npz")
            ABLATIONS[variant] = {"history": saved["history"], "best_epoch": saved["best_epoch"],
                                  "metrics": {**saved["validation"],
                                              "_per_impression": {"auc": per["auc"], "ndcg@5": per["ndcg5"]}}}
            log(f"--- ablation {variant}: reusing finished result")
            continue
        key = "shuffled" if variant == "attention_shuffled" else "ordered"
        if key not in TENSORS:
            ARRAYS[key] = make_arrays(variant)
            TENSORS[key] = {s: M.SplitTensors(ARRAYS[key][s], DEVICE) for s in ("train", "validation")}
        M.set_seed(SEED)
        ab_model = M.build_model(model_config(variant), ARTICLE_TENSORS, DEVICE)
        log(f"--- ablation: {variant}")
        hist, best = M.train_model(ab_model, TENSORS[key]["train"], TENSORS[key]["validation"],
                                   evaluate_validation, TRAIN_CONFIG, ab_dir,
                                   resume=bool(RESUME_RUN_ID),
                                   on_epoch_end=persist_training(ab_dir, f"ablations/{variant}"),
                                   log=log)
        scores = M.predict_logits(ab_model, TENSORS[key]["validation"])
        ABLATIONS[variant] = {"history": hist, "best_epoch": best["epoch"],
                              "metrics": evaluate_validation(scores, per_impression=True)}
        D.write_json({"variant": variant, "best_epoch": best["epoch"], "history": hist,
                      "validation": {k: v for k, v in ABLATIONS[variant]["metrics"].items()
                                     if not k.startswith("_")}}, ab_dir / "summary.json")
        per = ABLATIONS[variant]["metrics"]["_per_impression"]
        np.savez_compressed(ab_dir / "validation_per_impression.npz", auc=per["auc"], ndcg5=per["ndcg@5"])
        (ab_dir / "best_model_state.pt").unlink(missing_ok=True)   # metrics only; keep bundle small
        shutil.rmtree(ab_dir / "recovery", ignore_errors=True)
        if PERSIST_TO_DRIVE:   # remove the per-epoch copies of these from Drive too
            (PERSIST_RUN_DIR / "ablations" / variant / "best_model_state.pt").unlink(missing_ok=True)
            shutil.rmtree(PERSIST_RUN_DIR / "ablations" / variant / "recovery", ignore_errors=True)
        del ab_model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
else:
    print("Ablations skipped.")

## 6. Validation diagnostics and explainability

All numbers below are **development validation** results. Validation is used for
early stopping, selection and any tuning, so it is not an untouched test. Section 6
covers:
1. the selected model's metrics with counts and exclusions;
2. the two focused comparisons with paired user-cluster bootstrap intervals;
3. what the attention looks at: recency profile and same-category lift (a
   diagnostic, not a causal explanation);
4. performance by history length and for clicks on articles never referenced in
   training (cold content).

In [ ]:
VAL_SCORES = M.predict_logits(model, TENSORS[MAIN_KEY]["validation"])
VAL_METRICS = evaluate_validation(VAL_SCORES, per_impression=True)
per_imp = VAL_METRICS.pop("_per_impression")
log("Validation (development):", VAL_METRICS)

# Comparisons with user-cluster bootstrap (paired over the same impressions).
rows = [{"variant": MODEL_VARIANT + " (selected)", "best_epoch": BEST["epoch"],
         "val_auc": VAL_METRICS["mean_impression_auc"],
         "val_ndcg@5": VAL_METRICS["mean_impression_ndcg@5"]}]
COMPARISONS = {}
for variant, result in ABLATIONS.items():
    other = result["metrics"]["_per_impression"]
    boot = D.user_cluster_bootstrap(VAL.user_id, per_imp["auc"], other["auc"],
                                    BOOTSTRAP_SAMPLES, SEED)
    COMPARISONS[variant] = {"auc_difference_selected_minus_variant": boot}
    rows.append({"variant": variant, "best_epoch": result["best_epoch"],
                 "val_auc": result["metrics"]["mean_impression_auc"],
                 "val_ndcg@5": result["metrics"]["mean_impression_ndcg@5"],
                 "delta_auc_selected_minus_variant": boot["mean_difference"],
                 "ci95_low": boot["ci95"][0], "ci95_high": boot["ci95"][1]})
COMPARISON_TABLE = pd.DataFrame(rows)
COMPARISON_TABLE

In [ ]:
# AUC by history length and for "cold" clicked articles (never referenced in train).
lengths = VAL.hist_full_lengths[VAL.imp_user_row]
bins = [-1, 0, 9, 49, 199, np.inf]
labels_ = ["0", "1-9", "10-49", "50-199", "200+"]
bucket = pd.cut(lengths, bins=bins, labels=labels_)
by_length = (pd.DataFrame({"history_length": bucket, "auc": per_imp["auc"],
                           "ndcg@5": per_imp["ndcg@5"]})
             .groupby("history_length", observed=False)
             .agg(impressions=("auc", "size"), auc_eligible=("auc", "count"),
                  mean_auc=("auc", "mean"), mean_ndcg5=("ndcg@5", "mean")))
sizes = np.diff(VAL.offsets)
imp_of = np.repeat(np.arange(VAL.n_impressions), sizes)
clicked = VAL.labels == 1
cold = ~np.isin(VAL.cand_article_id, np.fromiter(seen_train_ids, dtype=np.int64))
cold_click_imp = np.zeros(VAL.n_impressions, bool)
np.logical_or.at(cold_click_imp, imp_of[clicked & cold], True)
by_cold = (pd.DataFrame({"clicked_article_unseen_in_train": cold_click_imp, "auc": per_imp["auc"]})
           .groupby("clicked_article_unseen_in_train")
           .agg(impressions=("auc", "size"), mean_auc=("auc", "mean")))
DIAGNOSTICS = {"by_history_length": by_length.reset_index().to_dict("records"),
               "by_cold_click": by_cold.reset_index().to_dict("records"),
               "validation_candidates_unseen_in_train": float(cold.mean())}
display(by_length)
display(by_cold)

In [ ]:
# Attention diagnostics on a fixed sample of validation impressions.
import matplotlib.pyplot as plt
rng = np.random.default_rng(SEED)
sample_imps = np.sort(rng.choice(VAL.n_impressions, size=min(2000, VAL.n_impressions), replace=False))
records = M.collect_attention(model, TENSORS[MAIN_KEY]["validation"], sample_imps)
ATTENTION_DIAG = None
if records:
    max_r = 20
    weight_by_recency, count_by_recency = np.zeros(max_r), np.zeros(max_r)
    uniform_by_recency = np.zeros(max_r)
    same_weight = same_uniform = 0.0
    for rec in records:
        w, L = rec["weights"], rec["hist_len"]
        cand_cat = ARTICLES.category[VAL.cand_article_row[rec["cand_index"]]]
        hist_cat = ARTICLES.category[rec["hist_rows"]]
        for r in range(max_r):
            pos = L - 1 - r
            ok = pos >= 0
            weight_by_recency[r] += w[np.flatnonzero(ok), pos[ok]].sum()
            uniform_by_recency[r] += (1.0 / L[ok]).sum()
            count_by_recency[r] += ok.sum()
        valid = np.arange(w.shape[1])[None, :] < L[:, None]
        match = valid & (hist_cat == cand_cat[:, None]) & (cand_cat[:, None] >= 3)
        has = L > 0
        same_weight += (w * match)[has].sum()
        same_uniform += (match.sum(1)[has] / L[has]).sum()
    ATTENTION_DIAG = {
        "impressions_sampled": int(len(sample_imps)),
        "mean_weight_by_recency_rank": (weight_by_recency / np.maximum(count_by_recency, 1)).tolist(),
        "uniform_weight_by_recency_rank": (uniform_by_recency / np.maximum(count_by_recency, 1)).tolist(),
        "same_category_attention_lift": float(same_weight / same_uniform) if same_uniform else None,
    }
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(range(1, max_r + 1), ATTENTION_DIAG["mean_weight_by_recency_rank"], marker="o",
            label="mean attention weight")
    ax.plot(range(1, max_r + 1), ATTENTION_DIAG["uniform_weight_by_recency_rank"], ls="--",
            label="uniform over history")
    ax.set_xlabel("history event recency rank (1 = most recent)")
    ax.set_ylabel("weight"); ax.legend(); ax.set_title("Validation attention by recency (diagnostic)")
    fig.tight_layout(); fig.savefig(RUN_DIR / "attention_recency.png", dpi=120); plt.show()
    log("Same-category attention lift (1 = no preference):", ATTENTION_DIAG["same_category_attention_lift"])
else:
    print("Selected variant has no attention weights.")

In [ ]:
# One explained ranking (diagnostic): top candidates and the history they attend to.
if records:
    titles = pd.concat([pd.read_parquet(p, columns=["article_id", "title", "category_str"])
                        for p in (PATHS["articles_train_val"], PATHS["articles_test"])]
                       ).drop_duplicates("article_id").set_index("article_id")
    rec = records[0]
    first_imp = sample_imps[0]
    lo, hi = VAL.offsets[first_imp], VAL.offsets[first_imp + 1]
    local = np.flatnonzero((rec["cand_index"] >= lo) & (rec["cand_index"] < hi))
    order = local[np.argsort(-rec["logits"][local])]
    rows = []
    for j in order[:5]:
        aid = int(VAL.cand_article_id[rec["cand_index"][j]])
        top = np.argsort(-rec["weights"][j])[:3]
        attended = [ARTICLES.article_ids[rec["hist_rows"][j][t]] for t in top if t < rec["hist_len"][j]]
        rows.append({"article_id": aid, "title": titles["title"].get(aid),
                     "category": titles["category_str"].get(aid),
                     "score": float(rec["logits"][j]),
                     "clicked": int(VAL.labels[rec["cand_index"][j]]),
                     "top_attended_history": [titles["title"].get(int(a)) for a in attended]})
    print(f"Validation behaviour {VAL.user_id[first_imp]}:{VAL.impression_id[first_imp]} "
          f"(history length {VAL.hist_lengths[VAL.imp_user_row[first_imp]]})")
    with pd.option_context("display.max_colwidth", 80):
        display(pd.DataFrame(rows))

**Interpretation (fill in after running).**
- **Attention vs `last_state`:** Δ AUC = … (95% CI …). Decision: keep target-aware
  attention only if the CI excludes 0 **and** the same-category lift is clearly
  > 1. Otherwise the simpler summary is sufficient.
- **Ordered vs shuffled:** Δ AUC = … (95% CI …). If the CI includes 0, the LSTM's
  order adds nothing beyond per-event recency, and a pooled model may be enough.
- **History length / cold clicks:** …
- **Adjustments made:** … (record any configuration change and why; do not iterate
  on validation for marginal gains).

## 7. Predict and export all splits

Scores are fp32 logits from the selected checkpoint, computed for every
candidate of every split.
- `score` is the logit, so `score_type = "logit"`.
- `click_probability = sigmoid(score)` comes from a BCE-trained model. It is
  uncalibrated and conditional on the logged slates; it is not site-wide CTR.
  Probabilities in a slate need not sum to 1.
- Train predictions are in-sample, not out-of-fold.
- The test split has no labels, so no test metric is computed.

In [ ]:
EXPORT_INFO, PREDICTIONS = {}, {}
for split in D.SPLITS:
    arrays = ARRAYS[MAIN_KEY][split]
    logits = M.predict_logits(model, TENSORS[MAIN_KEY][split])
    assert np.isfinite(logits).all(), split
    probs = 1.0 / (1.0 + np.exp(-logits)) if EXPORT_CLICK_PROBABILITY else None
    scores = D.score_frame(arrays, logits, probs)
    rankings = D.ranking_frame(scores)
    EXPORT_INFO[split] = D.validate_exports(arrays, scores, rankings)
    D.write_csv_atomic(scores, RUN_DIR / "predictions" / f"{split}_scores.csv")
    D.write_csv_atomic(rankings, RUN_DIR / "rankings" / f"{split}_ranked_articles.csv")
    PREDICTIONS[split] = logits
    log(split, "export:", EXPORT_INFO[split])
# Read-back check of one file (CSV round trip keeps full float precision).
check = pd.read_csv(RUN_DIR / "predictions" / "validation_scores.csv",
                    dtype={"behaviour_id": str, "article_id": "int64"},
                    float_precision="round_trip")
ref = D.score_frame(VAL, PREDICTIONS["validation"])
assert np.array_equal(check["score"].to_numpy(), ref["score"].to_numpy())
first_ranking = pd.read_csv(RUN_DIR / "rankings" / "validation_ranked_articles.csv").iloc[0]
print(first_ranking["behaviour_id"], json.loads(first_ranking["article_ids"])[:5], "...")

In [ ]:
# Model bundle, metrics, run configuration and dependencies.
MODEL_DIR = RUN_DIR / "model"
(MODEL_DIR / "code").mkdir(parents=True, exist_ok=True)
M.atomic_torch_save(model.state_dict(), MODEL_DIR / "model_state.pt")
D.write_json(MAIN_CONFIG.to_json(), MODEL_DIR / "model_config.json")
D.write_json(PREPROCESSING, MODEL_DIR / "preprocessing.json")
np.savez_compressed(
    MODEL_DIR / "mappings.npz",
    article_table_ids=ARTICLES.article_ids,
    id_vocab_ids=np.array(list(ID_VOCAB.keys()), dtype=np.int64),
    id_vocab_rows=np.array(list(ID_VOCAB.values()), dtype=np.int64),
    train_referenced_ids=np.array(sorted(seen_train_ids), dtype=np.int64))
for name in ("lstm_model.py", "lstm_data.py"):
    shutil.copy2(CODE_DIR / name, MODEL_DIR / "code" / name)
if TRAIN_MODEL:
    (RUN_DIR / "training_history.json").write_text(json.dumps(HISTORY, indent=2))
    pd.DataFrame(HISTORY).to_csv(RUN_DIR / "training_history.csv", index=False)
    recovery_src = RUN_DIR / "model_training" / "recovery" / "last_checkpoint.pt"
    if recovery_src.is_file():
        (MODEL_DIR / "recovery").mkdir(exist_ok=True)
        shutil.copy2(recovery_src, MODEL_DIR / "recovery" / "last_checkpoint.pt")
else:
    shutil.copy2(PERSIST_ROOT / LOAD_RUN_ID / "training_history.json", RUN_DIR / "training_history.json")

requirements = [f"torch=={torch.__version__}", f"numpy=={np.__version__}",
                f"pandas=={pd.__version__}", f"pyarrow=={pa.__version__}"]
try:
    import matplotlib; requirements.append(f"matplotlib=={matplotlib.__version__}")
except ImportError:
    pass
if TUNING_SUMMARY:
    requirements.append(f"optuna=={TUNING_SUMMARY['optuna_version']}")
(RUN_DIR / "requirements.txt").write_text("\n".join(requirements) + "\n")

METRICS = {
    "experiment_id": EXPERIMENT_ID, "run_id": RUN_ID,
    "partition": "validation (development: used for early stopping/selection"
                 + ("/tuning" if TUNING_SUMMARY else "") + "; not an untouched test)",
    "validation": VAL_METRICS, "metric_definitions": D.METRIC_DEFINITIONS,
    "selection": {"objective": "mean_impression_auc", "direction": "maximize",
                  "selected_epoch": BEST["epoch"], "tie_breaker": "earlier epoch"},
    "comparisons": {"table": COMPARISON_TABLE.to_dict("records"), "bootstrap": COMPARISONS,
                    "note": "single seed; paired user-cluster bootstrap over validation impressions"},
    "diagnostics": DIAGNOSTICS, "attention": ATTENTION_DIAG,
    "test": "unlabelled; no metric reported",
    "development_sample": SAMPLING or None, "mini_sample": IS_MINI_SAMPLE,
}
D.write_json(METRICS, RUN_DIR / "metrics.json")

SOURCE_RUN_CONFIG = (json.loads((PERSIST_ROOT / LOAD_RUN_ID / "run_config.json").read_text())
                     if not TRAIN_MODEL else None)
RUN_CONFIG = {
    "experiment_id": EXPERIMENT_ID, "run_id": RUN_ID, "owner": OWNER, "seed": SEED,
    "created": time.strftime("%Y-%m-%dT%H:%M:%S%z"), "mode": "train" if TRAIN_MODEL else "load",
    "source_run_id": None if TRAIN_MODEL else LOAD_RUN_ID, "smoke_test": SMOKE_TEST,
    "mini_sample": IS_MINI_SAMPLE, "development_sample": SAMPLING or None,
    "versions": VERSIONS, "code": GIT_INFO,
    "data_root": str(DATA_ROOT), "inputs": D.describe_inputs(PATHS, HASH_INPUTS),
    "path_notes": PATH_NOTES, "feature_case": FEATURE_CASE,
    "feature_table_columns_used": D.BASE_FEATURE_COLUMNS + DEVICE_COLUMNS,
    "engineered_aggregates_used": False,
    "input_tables": TABLE_INFO, "catalogue": CATALOGUE_INFO, "article_table": ARTICLE_INFO,
    "vectors": VECTOR_INFO,
    "history": {s: HISTORIES[s].stats for s in D.SPLITS},
    "history_policy": {
        "source": "each split's fixed pre-period history.parquet; no in-period behaviour appended",
        "cutoff": "every retained event strictly before the split's earliest impression (asserted)",
        "order": "chronological; original list position breaks timestamp ties; repeat reads kept",
        "truncation": f"most recent {HISTORY_MAX_LEN} events, oldest first, right-padded with masks",
        "empty_history": "learned empty-context vector + history_empty flag",
        "unknown_articles": "kept with missing/unknown category/type indices and in_catalogue=0",
        "missing_vectors": "zero placeholder input + has_text/has_image flag; rows kept",
    },
    "features": {"context": ARRAYS["ordered"]["train"].context_names,
                 "candidate": ARRAYS["ordered"]["train"].candidate_names,
                 "event": ARRAYS["ordered"]["train"].event_names,
                 "article_numeric": ARTICLES.numeric_names,
                 "excluded_for_leakage": ["behaviours read_time/scroll_percentage",
                                          "next_read_time/next_scroll_percentage",
                                          "total_inviews/total_pageviews/total_read_time"]},
    "preprocessing": {"fitted_on": "train" + (" (development sample)" if SAMPLING else ""),
                      "file": "model/preprocessing.json"},
    "model": {"config": MAIN_CONFIG.to_json(), **M.count_parameters(model),
              "frozen": ["text vectors (gemini-embedding-2, 768-d)", "image vectors (EB-NeRD, 1024-d)"],
              "trained": ["all embeddings, projections, LSTM, attention, head"]},
    "training": SOURCE_RUN_CONFIG["training"] if not TRAIN_MODEL else {"config": dict(TRAIN_HPARAMS), "loss": "BCEWithLogitsLoss over all candidates",
                 "optimizer": "AdamW", "scheduler": "ReduceLROnPlateau(max val AUC)",
                 "selected_epoch": BEST["epoch"], "seconds": TRAIN_SECONDS,
                 "ablations": {v: {"best_epoch": r["best_epoch"]} for v, r in ABLATIONS.items()}},
    "tuning": TUNING_SUMMARY,
    "exports": EXPORT_INFO, "export_schema_version": D.EXPORT_SCHEMA_VERSION,
    "score": {"score_type": "logit", "source": "selected checkpoint, fp32 inference",
              "click_probability": "sigmoid(score) from BCE training; uncalibrated, conditional "
              "on logged slates; probabilities in a slate need not sum to 1"
              if EXPORT_CLICK_PROBABILITY else None,
              "train_predictions": "in-sample (not out-of-fold)"},
    "persistent_location": str(PERSIST_RUN_DIR) if PERSIST_TO_DRIVE else None,
}
D.write_json(RUN_CONFIG, RUN_DIR / "run_config.json")
log("Bundle written to", RUN_DIR)

In [ ]:
# Persist the complete run folder to Drive and verify sizes + hashes.
PERSIST_RESULT = None
if PERSIST_TO_DRIVE:
    D.write_json(D.file_manifest(RUN_DIR), RUN_DIR / "manifest.json")
    PERSIST_RESULT = D.persist_directory(RUN_DIR, PERSIST_RUN_DIR)
    log("Persisted:", {k: v for k, v in PERSIST_RESULT.items() if k != "manifest"})
    assert PERSIST_RESULT["verified"], "Drive copy differs from the local run folder"
else:
    log("PERSIST_TO_DRIVE=False: the run exists only on this runtime; persistence incomplete.")

## 8. Reload check

This check uses only the **persistent copy**. It:
1. imports the bundled code;
2. rebuilds the model from `model_config.json` and loads `model_state.pt` with
   `weights_only=True` (no retraining);
3. rebuilds the inputs with the bundled `preprocessing.json` (no refitting);
4. predicts a fixed sample of validation and test impressions;
5. compares the predictions with the exported CSV scores read back from Drive.

The result is written to `reload_check.json`. It counts as evidence only after
this cell has actually run and printed `passed: True`.

In [ ]:
source_dir = PERSIST_RUN_DIR if PERSIST_TO_DRIVE else RUN_DIR
bundle = source_dir / "model"
spec_m = importlib.util.spec_from_file_location("bundled_lstm_model", bundle / "code" / "lstm_model.py")
spec_d = importlib.util.spec_from_file_location("bundled_lstm_data", bundle / "code" / "lstm_data.py")
BM, BD = importlib.util.module_from_spec(spec_m), importlib.util.module_from_spec(spec_d)
sys.modules[spec_m.name], sys.modules[spec_d.name] = BM, BD   # needed by @dataclass
spec_m.loader.exec_module(BM); spec_d.loader.exec_module(BD)

prep = json.loads((bundle / "preprocessing.json").read_text())
cfg = BM.ModelConfig.from_json(json.loads((bundle / "model_config.json").read_text()))
maps = np.load(bundle / "mappings.npz")
id_vocab = dict(zip(maps["id_vocab_ids"].tolist(), maps["id_vocab_rows"].tolist()))
reload_articles, _ = BD.build_article_table(
    CATALOGUE, all_ids, prep["feature_vocabulary"], prep["article_stats"], TEXT_LOOKUP,
    IMAGE_LOOKUP, prep["text_dim"], prep["image_dim"], prep["max_topics"], id_vocab)
reloaded = BM.build_model(cfg, BM.article_table_tensors(reload_articles, DEVICE), DEVICE)
reloaded.load_state_dict(torch.load(bundle / "model_state.pt", map_location=DEVICE, weights_only=True))
reloaded.eval()

RELOAD = {"source": str(bundle), "tolerance_abs_logit": RELOAD_TOLERANCE, "splits": {},
          "article_table_matches_saved": bool(np.array_equal(maps["article_table_ids"],
                                                             reload_articles.article_ids))}
for split in ("validation", "test"):
    hist = HISTORIES[split]
    if cfg.variant == "attention_shuffled":
        hist = BD.shuffled_history(hist, prep["shuffle_seed"] + BD.SPLITS.index(split))
    arrays = BD.build_split_arrays(split, FRAMES[split], OFFSETS[split], hist, reload_articles,
                                   prep["numeric_stats"], prep["device_columns"],
                                   disable_order_features=cfg.variant == "attention_shuffled")
    subset = np.sort(np.random.default_rng(SEED).choice(
        arrays.n_impressions, size=min(RELOAD_CHECK_IMPRESSIONS, arrays.n_impressions), replace=False))
    logits = BM.predict_logits(reloaded, BM.SplitTensors(arrays, DEVICE), impression_subset=subset)
    fresh = BD.score_frame(arrays, logits)
    fresh = fresh[fresh["score"].notna()]
    exported = pd.read_csv(source_dir / "predictions" / f"{split}_scores.csv",
                           dtype={"behaviour_id": str, "article_id": "int64"},
                    float_precision="round_trip")
    merged = fresh.merge(exported, on=["behaviour_id", "article_id"],
                                           suffixes=("_reload", "_export"), validate="one_to_one")
    diff = np.abs(merged["score_reload"] - merged["score_export"]).to_numpy()
    RELOAD["splits"][split] = {"impressions": int(len(subset)), "candidates": int(len(merged)),
                               "max_abs_diff": float(diff.max()), "mean_abs_diff": float(diff.mean())}
RELOAD["passed"] = RELOAD["article_table_matches_saved"] and all(v["max_abs_diff"] <= RELOAD_TOLERANCE and v["candidates"] > 0
                       for v in RELOAD["splits"].values())
D.write_json(RELOAD, RUN_DIR / "reload_check.json")
if PERSIST_TO_DRIVE:
    shutil.copy2(RUN_DIR / "reload_check.json", PERSIST_RUN_DIR / "reload_check.json")
log("Reload check:", RELOAD)
assert RELOAD["passed"], "reloaded predictions differ from exported scores beyond tolerance"

## 9. Business interpretation and handoff

**Fill in after the run, from Sections 6–8.**
- **Validation finding:** mean impression AUC … / nDCG@5 … (development data,
  selected epoch …).
- **Business link:** the score ranks every recorded candidate by estimated click
  relevance. The candidate-specific attention shows *which* past reading drives each
  recommendation, which helps the reranking owner explain topical concentration.
  Read-time and scroll inputs represent depth of past engagement. That does not
  establish continued reading.
- **Limits:**
  - These are offline click metrics on logged slates.
  - Validation was used for selection.
  - Single seed.
  - Histories are fixed pre-period snapshots: users' in-week reads are not used.
  - Test labels are unavailable.
  - Probabilities are uncalibrated.
- **For the reranking/evaluation owner:**
  - All three `*_scores.csv` files (`score` = logit, plus `click_probability`) and
    the three `*_ranked_articles.csv` files are under the persistent run folder
    printed below (export schema v2).
  - Join labels with `(behaviour_id, article_id)`.
  - Reload instructions: set `TRAIN_MODEL = False` and `LOAD_RUN_ID` to this run.

In [ ]:
# Small files to return for review (no datasets or model binaries).
RETURN_FILES = ["metrics.json", "run_config.json", "training_history.csv", "reload_check.json",
                "requirements.txt", "logs/notebook_log.txt", "attention_recency.png",
                "manifest.json"]
RETURN_FILES += [f"ablations/{v}/summary.json" for v in ABLATIONS]
review_zip = RUN_DIR / f"review_{RUN_ID}.zip"
import zipfile
with zipfile.ZipFile(review_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in RETURN_FILES:
        path = RUN_DIR / name
        if path.is_file():
            zf.write(path, name)
    if TUNING_SUMMARY:
        trials = Path(TUNING_SUMMARY["trials_csv"])
        if trials.is_file():
            zf.write(trials, "tuning/trials.csv")
if PERSIST_TO_DRIVE:
    shutil.copy2(review_zip, PERSIST_RUN_DIR / review_zip.name)
    # Refresh the manifest so it also covers reload_check.json and the review zip.
    D.write_json(D.file_manifest(RUN_DIR), RUN_DIR / "manifest.json")
    final = D.persist_directory(RUN_DIR, PERSIST_RUN_DIR)
    log("Final persistence verified:", final["verified"], "| files:", final["files"])
print("Persistent run folder:", PERSIST_RUN_DIR if PERSIST_TO_DRIVE else "(not persisted)")
print("Review bundle to return with the executed notebook:",
      (PERSIST_RUN_DIR if PERSIST_TO_DRIVE else RUN_DIR) / review_zip.name)
print(COMPARISON_TABLE.to_string(index=False))